<a href="https://colab.research.google.com/github/PRANAV-MAGARDE/Odyssey/blob/main/Unified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!cp -r '/content/drive/MyDrive/hubble_dataset' '/content'

KeyboardInterrupt: 

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = 'erayarising'
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_f5824c6bfc9fbe40d6dcc318bd6c7ccc'
%cd /content
!kaggle competitions download -c galaxy-zoo-the-galaxy-challenge

!unzip galaxy-zoo-the-galaxy-challenge.zip && unzip images_training_rev1.zip && rm *.zip


# def clean_data(input_csv, clean_galaxy_labels, threshold):
#     input= pd.read_csv(input_csv)
#     input = input[['GalaxyID', 'Class1.1', 'Class1.2', 'Class1.3']] #Class 1.1 for elliptical and 1.2 for spiral galaxies; .3 was for uncertain/ stars

#     mask = input['Class1.3'] < 0.5
#     input = input[mask]
#     input['Label'] = -1

#     input.loc[input['Class1.2'] >= threshold, 'Label'] = 1
#     input.loc[input['Class1.1'] >= threshold, 'Label'] = 0
#     clean = input[input['Label'] != -1].copy()
#     final = clean[['GalaxyID', 'Label']]
#     final.to_csv(clean_galaxy_labels, index=False)
#     return clean_galaxy_labels

# clean_data(input_csv= '/content/drive/MyDrive/training_solutions_rev1.csv',clean_galaxy_labels= 'clean_galaxy_labels.csv', threshold=0.8)

Streaming output truncated to the last 5000 lines.
  inflating: images_training_rev1/926425.jpg  
  inflating: images_training_rev1/926426.jpg  
  inflating: images_training_rev1/926435.jpg  
  inflating: images_training_rev1/926446.jpg  
  inflating: images_training_rev1/926448.jpg  
  inflating: images_training_rev1/926453.jpg  
  inflating: images_training_rev1/926462.jpg  
  inflating: images_training_rev1/926478.jpg  
  inflating: images_training_rev1/926480.jpg  
  inflating: images_training_rev1/926484.jpg  
  inflating: images_training_rev1/926488.jpg  
  inflating: images_training_rev1/926509.jpg  
  inflating: images_training_rev1/926522.jpg  
  inflating: images_training_rev1/926544.jpg  
  inflating: images_training_rev1/926547.jpg  
  inflating: images_training_rev1/926548.jpg  
  inflating: images_training_rev1/926551.jpg  
  inflating: images_training_rev1/926602.jpg  
  inflating: images_training_rev1/926630.jpg  
  inflating: images_training_rev1/926649.jpg  
  inflati

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

# ── Label mapping ─────────────────────────────────────────────
# 0=elliptical, 1=spiral, 2=nebula, 3=star_cluster, 4=planetary

HUBBLE_BASE = "/content/drive/MyDrive/hubble_dataset"

hubble_category_map = {
    "galaxies"    : 1,  # default spiral, refine later if needed
    "nebulae"     : 2,
    "star_clusters": 3,
    "solar_system": 4,  # planetary objects
}

# ── 1. Build Hubble DataFrame ─────────────────────────────────
def build_hubble_df(base_dir, category_map):
    records = []
    for category, label in category_map.items():
        cat_dir = os.path.join(base_dir, category)
        if not os.path.exists(cat_dir):
            print(f"Missing: {cat_dir}")
            continue
        for fname in os.listdir(cat_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                records.append({
                    'img_path': os.path.join(cat_dir, fname),
                    'Label'   : label
                })
    df = pd.DataFrame(records)
    print(f"Hubble: {len(df)} images")
    print(df['Label'].value_counts())
    return df

hubble_df = build_hubble_df(HUBBLE_BASE, hubble_category_map)


# ── 2. Build Galaxy Zoo DataFrame ────────────────────────────
galaxyzoo_df = pd.read_csv('clean_galaxy_labels.csv')
galaxyzoo_df['img_path'] = galaxyzoo_df['GalaxyID'].apply(
    lambda x: f'/content/images_training_rev1/{int(x)}.jpg'
)
galaxyzoo_df = galaxyzoo_df[['img_path', 'Label']]
print(f"\nGalaxy Zoo: {len(galaxyzoo_df)} images")
print(galaxyzoo_df['Label'].value_counts())

# ── 3. Build DeepSky DataFrame ────────────────────────────────
# deepsky_df = pd.read_csv('/content/drive/MyDrive/deepsky_labeled.csv')[['img_path', 'Label']]
# print(f"\nDeepSky: {len(deepsky_df)} images")
# print(deepsky_df['Label'].value_counts())

# ── 4. Unify ──────────────────────────────────────────────────
unified_df = pd.concat([galaxyzoo_df, hubble_df], ignore_index=True)

# filter only existing images
unified_df = unified_df[unified_df['img_path'].apply(os.path.exists)].reset_index(drop=True)

# shuffle
unified_df = unified_df.sample(frac=1, random_state=42).reset_index(drop=True)
unified_df.to_csv('unified_dataset.csv', index=False)

label_names = {0:'elliptical', 1:'spiral', 2:'nebula', 3:'star_cluster', 4:'planetary'}
print(f"\nTotal unified: {len(unified_df)}")
print(unified_df['Label'].map(label_names).value_counts())

Hubble: 3696 images
Label
1    1853
4     829
2     701
3     313
Name: count, dtype: int64

Galaxy Zoo: 24273 images
Label
1    16141
0     8132
Name: count, dtype: int64

Total unified: 27969
Label
spiral          17994
elliptical       8132
planetary         829
nebula            701
star_cluster      313
Name: count, dtype: int64


In [ ]:
# balance — cap majority classes to 2000 max
balanced_df = unified_df.groupby('Label').apply(
    lambda x: x.sample(min(len(x), 1200), random_state=42)
).reset_index(drop=True)

balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
balanced_df.to_csv('balanced_dataset.csv', index=False)

print(f"\nBalanced: {len(balanced_df)}")
print(balanced_df['Label'].map(label_names).value_counts())


Balanced: 4243
Label
elliptical      1200
spiral          1200
planetary        829
nebula           701
star_cluster     313
Name: count, dtype: int64


/tmp/ipykernel_15020/1643295761.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = unified_df.groupby('Label').apply(


In [ ]:
import shutil
import os

DRIVE_OUTPUT = "/content/drive/MyDrive/unified_dataset"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# ── 1. Save CSV ───────────────────────────────────────────────
unified_df.to_csv(f"{DRIVE_OUTPUT}/unified_dataset.csv", index=False)
balanced_df.to_csv(f"{DRIVE_OUTPUT}/balanced_dataset.csv", index=False)
print("CSVs saved!")

# ── 2. Copy images to Drive ───────────────────────────────────
images_output = os.path.join(DRIVE_OUTPUT, "images")
os.makedirs(images_output, exist_ok=True)

label_names = {0:'elliptical', 1:'spiral', 2:'nebula', 3:'star_cluster', 4:'planetary'}

# create subfolders per class
for label, name in label_names.items():
    os.makedirs(os.path.join(images_output, name), exist_ok=True)

# copy each image
total = len(balanced_df)
for idx, row in balanced_df.iterrows():
    src      = row['img_path']
    label    = row['Label']
    class_name = label_names[label]
    filename = os.path.basename(src)
    dst      = os.path.join(images_output, class_name, filename)

    if not os.path.exists(dst):
        shutil.copy2(src, dst)

    if (idx+1) % 500 == 0:
        print(f"  Copied {idx+1}/{total}...")

print("All images saved to Drive!")

# ── 3. Verify ─────────────────────────────────────────────────
for name in label_names.values():
    path  = os.path.join(images_output, name)
    count = len(os.listdir(path))
    print(f"{name}: {count} images")

CSVs saved!
  Copied 500/4243...
  Copied 1000/4243...
  Copied 1500/4243...
  Copied 2000/4243...
  Copied 2500/4243...
  Copied 3000/4243...
  Copied 3500/4243...
  Copied 4000/4243...
All images saved to Drive!
elliptical: 1200 images
spiral: 1200 images
nebula: 701 images
star_cluster: 313 images
planetary: 829 images


In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # makes CUDA errors show at exact line
!cp -r '/content/drive/MyDrive/unified_dataset' '/content'

cp: cannot open '/content/drive/MyDrive/unified_dataset/balanced_dataset.gsheet' for reading: Operation not supported


In [ ]:
import pandas as pd
df = pd.read_csv('/content/unified_dataset/unified_dataset.csv')
print(df['Label'].unique())   # should only show [0, 1, 2, 3, 4]
print(df['Label'].min())      # should be 0
print(df['Label'].max())      # should be 4
print(df['Label'].isna().sum()) # should be 0

[1 0 4 2 3]
0
4
0


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────
INPUT_CSV    = '/content/unified_dataset/balanced_dataset.csv'
OUTPUT_DIR   = '/content/unified_dataset/clahe_images'
OUTPUT_CSV   = '/content/unified_dataset/balanced_dataset_clahe.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── CLAHE setup ───────────────────────────────────────────────
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def apply_clahe(img_path):
    img     = Image.open(img_path).convert('RGB')
    img_np  = np.array(img)
    lab     = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l_new   = clahe.apply(l)
    merged  = cv2.merge((l_new, a, b))
    result  = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    return Image.fromarray(result)

# ── Process all images ────────────────────────────────────────
df      = pd.read_csv(INPUT_CSV)
records = []
failed  = 0

for idx, row in tqdm(df.iterrows(), total=len(df)):
    src_path = row['img_path']
    label    = row['Label']

    # generate output filename using index to avoid conflicts
    fname    = f"{idx}_{os.path.basename(src_path)}"
    dst_path = os.path.join(OUTPUT_DIR, fname)

    try:
        if os.path.exists(dst_path):
            records.append({'img_path': dst_path, 'Label': label})
            continue

        img = apply_clahe(src_path)

        # resize to 224 here too — saves time during training
        img = img.resize((224, 224), Image.LANCZOS)
        img.save(dst_path, quality=95)

        records.append({'img_path': dst_path, 'Label': label})

    except Exception as e:
        print(f"Failed: {src_path} — {e}")
        failed += 1

# ── Save new CSV ──────────────────────────────────────────────
new_df = pd.DataFrame(records)
new_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nDone! Processed: {len(new_df)} | Failed: {failed}")
print(f"New CSV saved: {OUTPUT_CSV}")

label_names = {0:'elliptical', 1:'spiral', 2:'nebula', 3:'star_cluster', 4:'planetary'}
print(new_df['Label'].map(label_names).value_counts())

 47%|████▋     | 1982/4243 [13:43<09:40,  3.89it/s]

In [ ]:
from typing import ChainMap
import torch
import torch.nn as nn
import pandas as pd
from torchvision import models, transforms
from PIL import Image, ImageFile
from google.colab import files
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os # Added os import
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True # Allow loading of truncated images
class ApplyCLAHE(object):
    def __init__(self, clip_limit=2.0, tile_grid_size=(8,8)): #init is a constructor: it runs whenever a new object is created
        self.clip_limit= clip_limit
        self.tile_grid_size= tile_grid_size
        self.clahe= None

    def __call__(self, img):
        if self.clahe is None:
            self.clahe= cv2.createCLAHE(self.clip_limit, self.tile_grid_size)

        img_np = np.array(img)
        lab_img = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l, a, b= cv2.split(lab_img)
        l_new = self.clahe.apply(l)
        lab_merged= cv2.merge((l_new, a, b))
        img_new = cv2.cvtColor(lab_merged, cv2.COLOR_LAB2RGB)

        return Image.fromarray(img_new)

astronomy_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    ApplyCLAHE(),

    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class UnifiedDataset(Dataset):
    def __init__(self, csv_file, transforms):
        self.unified_frame = pd.read_csv(csv_file)
        self.transform = transforms

    def __len__(self):
        return len(self.unified_frame)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_path = self.unified_frame.iloc[idx]['img_path']
        label = self.unified_frame.iloc[idx]['Label']
        image = Image.open(img_path).convert('RGB')

        image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct= 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc


if __name__ == '__main__':
    unified_dataset = UnifiedDataset(csv_file='/content/unified_dataset/unified_dataset.csv', transforms=astronomy_transforms)
    print("Data loaded!")

    train_size= int(0.8 * len(unified_dataset))
    test_size= len(unified_dataset) - train_size
    train_data, test_data = random_split(unified_dataset, [train_size, test_size])
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)


    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = models.efficientnet_b3(weights='IMAGENET1K_V1')
    for param in model.parameters():
        param.requires_grad = False
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 5)
    model = model.to(device)
    print("model done!")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

    print("training started")
    Epochs = 9
    for epoch in range(Epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)

        print(f"Epoch {epoch+1}/{Epochs} | "
              f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f} | ")

    torch.save(model.state_dict(), '/content/drive/MyDrive/efficientnet_odyssey.pth')
    print("Model Saved!")

Data loaded!
model done!
training started


KeyboardInterrupt: 

In [ ]:

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0,0
    with torch.no_grad():
        for images,labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1)==labels).sum().item()

        acc = correct / len(loader.dataset)
        return total_loss / len(loader), acc

         #   test_loader = DataLoader(test_data, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)